In [ ]:
!nvidia-smi

: 

In [ ]:
!pip install tensorflow

: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls

drive  sample_data


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model
import os

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [ ]:

# Path to the 'maize_dataset' folder
DATA_DIR = '/content/drive/MyDrive/GROW_SAFE/maize_dataset/train'

In [ ]:

# Image parameters
IMG_SIZE = 128
IMG_SHAPE = (128, 128)
BATCH_SIZE = 32

In [ ]:
input_tensor = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

In [ ]:
datagen = ImageDataGenerator(
    rescale=1./255, # Normalize pixel values
    validation_split=0.2 # Use 20% of data for validation
)

In [ ]:
# Load the training data
train_generator = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary', # We have two classes: 'good' and 'bad'
    subset='training'
)

In [ ]:
# Load the validation data
validation_generator = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

In [ ]:
# Load the MobileNetV2 base model
# 'weights=imagenet' uses pre-trained weights on a massive dataset
# 'include_top=False' removes the final classification layer, which we will replace

base_model = MobileNetV2(
    input_tensor=input_tensor,
    include_top=False,
    weights='imagenet'
)

In [ ]:
# Freeze the base model layers
# Prevents the pre-trained weights from being updated during training
base_model.trainable = False

In [ ]:
# Build the classification head for our maize problem
x = base_model.output

# Downsample the features
x = GlobalAveragePooling2D()(x)

x = Dense(128, activation='relu')(x)

# Sigmoid for binary classification (0 or 1)
output = Dense(1, activation='sigmoid')(x) 

In [ ]:

# Create the final model
model = Model(inputs=base_model.input, outputs=output)

# Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Define the path to your dataset in Google Drive
# NOTE: Replace 'YOUR_DRIVE_PATH' with the actual path to the 'maize_dataset' folder
DATA_DIR = '/content/drive/MyDrive/GROW_SAFE/maize_dataset/train'

# --- Configuration ---
IMG_SIZE = 128

# EXPLICITLY define the input tensor
# This is the critical step to ensure the shape is correctly serialized.
input_tensor = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

# Image parameters
IMG_SIZE = (128, 128) # Use 128x128 since the augmented data is this size
BATCH_SIZE = 32

# Create a data generator for training and validation
# We'll use 80% for training and 20% for validation
datagen = ImageDataGenerator(
    rescale=1./255, # Normalize pixel values
    validation_split=0.2 # Use 20% of data for validation
)

# Load the training data
train_generator = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary', # We have two classes: 'good' and 'bad'
    subset='training'
)

# Load the validation data
validation_generator = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

In [ ]:
# 1. Load the MobileNetV2 base model
# 'weights=imagenet' uses pre-trained weights on a massive dataset
# 'include_top=False' removes the final classification layer, which we will replace
base_model = MobileNetV2(
    input_tensor=input_tensor,
    include_top=False,
    weights='imagenet'
)

# 2. Freeze the base model layers
# This prevents the pre-trained weights from being updated during training
base_model.trainable = False

# 3. Build the classification head for our maize problem
x = base_model.output
x = GlobalAveragePooling2D()(x) # A simple way to downsample the features
x = Dense(128, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x) # Sigmoid for binary classification (0 or 1)

# 4. Create the final model
model = Model(inputs=base_model.input, outputs=output)

# 5. Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

# Define the path to save the checkpoints
CHECKPOINT_PATH = '/content/drive/MyDrive/GROW_SAFE/v2/maize_classifier_epoch{epoch:02d}_model.h5'

# Create a ModelCheckpoint callback
# Saves the model after each epoch
checkpoint_callback = ModelCheckpoint(
    filepath=CHECKPOINT_PATH,
    save_weights_only=False, # Set to True if you only want to save weights
    save_best_only=False, # Set to True to save only the best model based on a metric (e.g., 'val_accuracy')
    monitor='val_accuracy', # Metric to monitor if save_best_only is True
    verbose=1
)

# Train the model
# You can increase the number of epochs for better accuracy
EPOCHS = 1

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    validation_steps=validation_generator.samples // BATCH_SIZE,
    callbacks=[checkpoint_callback] # Add the callback here
)

# Save the model in the Keras SavedModel format (recommended)
# This will save it to your Google Drive
# MODEL_SAVE_PATH = '/content/drive/MyDrive/GROW_SAFE/v2/maize_classifier_model'
# model.save(MODEL_SAVE_PATH)

# print(f"Model saved to: {MODEL_SAVE_PATH}")

In [ ]:
from tensorflow.keras.callbacks import Callback

class AccurateCheckpoint(Callback):
    def __init__(self, checkpoint_path, initial_threshold=0.60, improvement_threshold=0.02):
        super(AccurateCheckpoint, self).__init__()
        self.checkpoint_path = checkpoint_path
        self.initial_threshold = initial_threshold
        self.improvement_threshold = improvement_threshold
        self.best_accuracy = 0

    def on_epoch_end(self, epoch, logs=None):
        current_accuracy = logs.get('accuracy')
        if current_accuracy is None:
            return

        # Check if the accuracy has reached the initial threshold
        if current_accuracy >= self.initial_threshold:
            # Check if the current accuracy is a significant improvement over the best accuracy
            if current_accuracy >= self.best_accuracy + self.improvement_threshold:
                self.best_accuracy = current_accuracy
                filepath = self.checkpoint_path.format(epoch=epoch+1)
                self.model.save(filepath)
                print(f"\nEpoch {epoch+1}: Training accuracy improved to {current_accuracy:.4f}, saving model to {filepath}")
            else:
                print(f"\nEpoch {epoch+1}: Training accuracy {current_accuracy:.4f}, no significant improvement.")
        else:
            print(f"\nEpoch {epoch+1}: Training accuracy {current_accuracy:.4f}, below initial threshold.")

# Instantiate the custom callback
custom_checkpoint_callback = AccurateCheckpoint(
    checkpoint_path='/content/drive/MyDrive/GROW_SAFE/v2/maize_classifier_epoch{epoch:02d}_model.h5',
    initial_threshold=0.60,
    improvement_threshold=0.02
)

# Train the model with the custom callback
EPOCHS = 10 # You might want to increase epochs for better results

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    validation_steps=validation_generator.samples // BATCH_SIZE,
    callbacks=[custom_checkpoint_callback] # Use the custom callback here
)

In [ ]:
MODEL_SAVE_PATH = '/content/drive/MyDrive/SAFE_GROW/v2/maize_classifier_epoch1_model.h5'
model.save(MODEL_SAVE_PATH)
print(f"Model successfully saved to: {MODEL_SAVE_PATH}")

In [ ]:
!pip install tensorflowjs

In [ ]:
!tensorflowjs_converter --input_format keras /content/drive/MyDrive/maize_dataset/maize_classifier_epoch1_model.keras /content/drive/MyDrive/maize_dataset/tfjs_model_files

In [ ]:
!tensorflowjs_converter --version